STEP 1: Create Useful Features

In [1]:
import pandas as pd
transaction = pd.read_excel("Dataset/Transaction.xlsx")
user = pd.read_excel("Dataset/User.xlsx")
city = pd.read_excel("Dataset/City.xlsx")
country = pd.read_excel("Dataset/Country.xlsx")
region = pd.read_excel("Dataset/Region.xlsx")
continent = pd.read_excel("Dataset/Continent.xlsx")



In [2]:

print(transaction.head())



   TransactionId  UserId  VisitYear  VisitMonth  VisitMode  AttractionId  \
0              3   70456       2022          10          2           640   
1              8    7567       2022          10          4           640   
2              9   79069       2022          10          3           640   
3             10   31019       2022          10          3           640   
4             15   43611       2022          10          2           640   

   Rating  
0       5  
1       5  
2       5  
3       3  
4       3  


Recreate Your Final DataFrame

In [3]:
user_city = user.merge(city, on="CityId", how="left")

df = (
    user_city
    .merge(country, left_on="CountryId_x", right_on="CountryId", how="left")
    .merge(region, left_on="RegionId_y", right_on="RegionId", how="left")
    .merge(continent, left_on="ContinentId_y", right_on="ContinentId", how="left")
    .merge(transaction[["UserId", "Rating"]], 
              on="UserId", 
              how="left")

)
df = df.merge(transaction[["UserId", "AttractionId"]],
              on="UserId",
              how="left")



Create Useful Features

In [4]:
# User average rating
user_avg = df.groupby("UserId")["Rating"].mean().reset_index()
user_avg.rename(columns={"Rating": "UserAvgRating"}, inplace=True)

df = df.merge(user_avg, on="UserId", how="left")


STEP 2: Attraction Average Rating

In [5]:
attr_avg = df.groupby("AttractionId")["Rating"].mean().reset_index()
attr_avg.rename(columns={"Rating": "AttractionAvgRating"}, inplace=True)

df = df.merge(attr_avg, on="AttractionId", how="left")


STEP 3: Encoding Categorical Columns

In [47]:
# ==============================
# 1️⃣ Load Attraction File
# ==============================
attraction = pd.read_excel(
    "Dataset/Additional_Data_for_Attraction_Sites/Updated_Item.xlsx"
)

# ==============================
# 2️⃣ Merge Attraction Info
# ==============================
df = df.merge(
    attraction[["AttractionId", "AttractionTypeId"]],
    on="AttractionId",
    how="left"
)

# ==============================
# 3️⃣ Encode Categorical Columns (Safe Version)
# ==============================

from sklearn.preprocessing import LabelEncoder

categorical_cols = ["Continent", "Country", "AttractionTypeId", "VisitMode"]

encoders = {}

for col in categorical_cols:
    
    if col in df.columns:   # 🔥 prevents KeyError
        
        # Convert to string and handle missing values
        df[col] = df[col].astype(str).fillna("Unknown")
        
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        encoders[col] = le
        
        print(f"{col} encoded successfully.")
        
    else:
        print(f"{col} not found in dataframe. Skipping.")

print("\nEncoding Completed ✅")


Continent encoded successfully.
Country encoded successfully.
AttractionTypeId encoded successfully.
VisitMode not found in dataframe. Skipping.

Encoding Completed ✅


STEP 4: Save Final Dataset

In [6]:
df.to_csv("dataset/final_model_data.csv", index=False)
